# Dependencies

In [2]:
# Colab setup ------------------
import os, sys, subprocess
if "google.colab" in sys.modules:
    cmd = "pip install --upgrade iqplot bebi103 arviz cmdstanpy watermark"
    process = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    stdout, stderr = process.communicate()
    import cmdstanpy; cmdstanpy.install_cmdstan()
    data_path = "https://s3.amazonaws.com/bebi103.caltech.edu/data/"
else:
    data_path = "../data/"
# ------------------------------

DEBUG:cmdstanpy:cmd: make examples/bernoulli/bernoulli
cwd: None


CmdStan install directory: /root/.cmdstan
CmdStan version 2.36.0 already installed
Test model compilation


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Imports and Functions

In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

In [5]:
def predict_value(theta, mu, sigma, alpha, beta):
    mixture_component = np.random.binomial(1, theta)
    val = np.where(mixture_component, np.random.gamma(alpha,beta), np.random.normal(mu, sigma))
    return float(max(0,val))

In [6]:
def traceplot(modelparam, nchain, niter, ax, name):
  for i in range(nchain):
    ax.plot(modelparam[i*niter:(i+1)*niter], label = i+1)
  #ax.legend()
  ax.set_ylabel(name)
  #plt.show()

# Data

In [17]:
df_habits = pd.read_csv("/content/drive/My Drive/dev_lastANDtest.csv")

In [18]:
df_habits.describe()

,last,dev,dev_percent,Pilot,Value
count,163.000000,163.000000,163.000000,163.000000,163.000000
mean,0.872122,0.445906,0.506313,2.852761,0.506323
std,0.119010,0.416361,0.471808,1.393251,0.471808
min,0.448980,0.000000,0.000000,1.000000,0.000010
25%,0.840455,0.000000,0.000000,2.000000,0.000010
50%,0.893617,0.508772,0.588889,3.000000,0.588899
75%,0.950000,0.878283,0.970252,4.000000,0.970262
max,1.041667,1.000000,1.284349,5.000000,1.284359


# Bayes Factor

In [19]:
import numpy as np
import matplotlib.pyplot as plt
from cmdstanpy import CmdStanModel
from scipy.stats import gaussian_kde, norm

# Compile and fit the Stan model
model_code = """
data {
  int<lower=0> N_A;
  int<lower=0> N_B;
  vector[N_A] y_A;
  vector[N_B] y_B;
}
parameters {
  real mu_A;
  real mu_B;
  real<lower=0> sigma_A;
  real<lower=0> sigma_B;
}
model {
  mu_A ~ normal(0, 1);
  mu_B ~ normal(0, 1);
  sigma_A ~ cauchy(2,2);
  sigma_B ~ cauchy(2,2);

  y_A ~ normal(mu_A, sigma_A);
  y_B ~ normal(mu_B, sigma_B);
}
generated quantities {
  real diff;
  diff = mu_A - mu_B;
}
"""

# Save the Stan model code to a file
with open("/content/drive/My Drive/Stans/bayes_mannwhitney.stan", "w") as f:
    f.write(model_code)

# Compile the Stan model
model = CmdStanModel(stan_file="/content/drive/My Drive/Stans/bayes_mannwhitney.stan")

06:43:17 - cmdstanpy - INFO - compiling stan file /tmp/tmp43hsn3te/tmp0b6gjlzr.stan to exe file /content/drive/My Drive/Stans/bayes_mannwhitney
INFO:cmdstanpy:compiling stan file /tmp/tmp43hsn3te/tmp0b6gjlzr.stan to exe file /content/drive/My Drive/Stans/bayes_mannwhitney
DEBUG:cmdstanpy:cmd: make STANCFLAGS+=--filename-in-msg=bayes_mannwhitney.stan /tmp/tmp43hsn3te/tmp0b6gjlzr
cwd: /root/.cmdstan/cmdstan-2.36.0
DEBUG:cmdstanpy:Console output:

--- Translating Stan model to C++ code ---
bin/stanc --filename-in-msg=bayes_mannwhitney.stan --o=/tmp/tmp43hsn3te/tmp0b6gjlzr.hpp /tmp/tmp43hsn3te/tmp0b6gjlzr.stan

--- Compiling C++ code ---
g++ -std=c++17 -pthread -D_REENTRANT -Wno-sign-compare -Wno-ignored-attributes -Wno-class-memaccess      -I stan/lib/stan_math/lib/tbb_2020.3/include    -O3 -I src -I stan/src -I stan/lib/rapidjson_1.1.0/ -I lib/CLI11-1.9.1/ -I stan/lib/stan_math/ -I stan/lib/stan_math/lib/eigen_3.4.0 -I stan/lib/stan_math/lib/boost_1.84.0 -I stan/lib/stan_math/lib/sundial

## By Pilot

In [22]:
d = {'Pilot': [], 'BF01': [], 'p_greater': []}
for i in range(5):
  # Simulated data (group A and group B)

  group_A = df_habits[(df_habits['Group'] == 'ET') & (df_habits['Pilot'] == i+1)]['dev_percent']
  group_B = df_habits[(df_habits['Group'] == 'ST') & (df_habits['Pilot'] == i+1)]['dev_percent']

  # Prepare the data dictionary for Stan
  data = {
      'N_A': len(group_A),
      'N_B': len(group_B),
      'y_A': group_A,
      'y_B': group_B
  }

  fit = model.sample(data=data, chains=4, parallel_chains=4)

  # Extract the posterior samples for the difference
  diff_samples = fit.stan_variable('diff')

  # Posterior KDE for the difference
  kde = gaussian_kde(diff_samples)
  x_range = np.linspace(-3, 3, 1000)
  posterior_density = kde.evaluate(x_range)

  # Prior for the difference (Normal(0, sqrt(2)))
  prior_density = norm.pdf(x_range, loc=0, scale=np.sqrt(2))

  # Bayes Factor using Savage-Dickey
  p_post_0 = kde.evaluate(0)[0]
  p_prior_0 = norm.pdf(0, loc=0, scale=np.sqrt(2))
  BF_01 =  p_post_0 / p_prior_0

  p_greater = np.mean(diff_samples > 0)

  # Print Bayes Factor
  d['Pilot'].append(i+1)
  d['BF01'].append(BF_01)
  d['p_greater'].append(p_greater)

DEBUG:cmdstanpy:cmd: /content/drive/My Drive/Stans/bayes_mannwhitney info
cwd: None
DEBUG:cmdstanpy:input tempfile: /tmp/tmpadqwb6x8/6iczmooq.json
06:43:59 - cmdstanpy - INFO - CmdStan start processing
INFO:cmdstanpy:CmdStan start processing


chain 1 |          | 00:00 Status

chain 2 |          | 00:00 Status

chain 3 |          | 00:00 Status

chain 4 |          | 00:00 Status

...

chain 2 |          | 00:00 Status

chain 3 |          | 00:00 Status

chain 4 |          | 00:00 Status

DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: 1
DEBUG:cmdstanpy:CmdStan args: ['/content/drive/My Drive/Stans/bayes_mannwhitney', 'id=1', 'random', 'seed=63674', 'data', 'file=/tmp/tmpadqwb6x8/njpjf373.json', 'output', 'file=/tmp/tmpadqwb6x8/bayes_mannwhitney4d26hpof/bayes_mannwhitney-20250423064402_1.csv', 'method=sample', 'algorithm=hmc', 'adapt', 'engaged=1']
DEBUG:cmdstanpy:idx 1
DEBUG:cmdstanpy:idx 2
DEBUG:cmdstanpy:running CmdStan, num_threads: 1
DEBUG:cmdstanpy:CmdStan args: ['/content/drive/My Drive/Stans/bayes_mannwhitney', 'id=3', 'random', 'seed=63674', 'data', 'file=/tmp/tmpadqwb6x8/njpjf373.json', 'output', 'file=/tmp/tmpadqwb6x8/bayes_mannwhitney4d26hpof/bayes_mannwhitney-20250423064402_3.csv', 'method=sample', 'algorithm=hmc', 'adapt', 'engaged=1']
DEBUG:cmdstanpy:running CmdStan, num_threads: 1
DEBUG:cmdstanpy:CmdStan args: ['/content/drive/My Drive/Stans/bayes_mannwhitney', 'id=2', 'random', 'seed=63674', 'data', 'file=/tmp/tmpadqwb6x8/njpjf373.js

06:44:02 - cmdstanpy - INFO - CmdStan done processing.
INFO:cmdstanpy:CmdStan done processing.
DEBUG:cmdstanpy:runset
RunSet: chains=4, chain_ids=[1, 2, 3, 4], num_processes=4
 cmd (chain 1):
	['/content/drive/My Drive/Stans/bayes_mannwhitney', 'id=1', 'random', 'seed=63674', 'data', 'file=/tmp/tmpadqwb6x8/njpjf373.json', 'output', 'file=/tmp/tmpadqwb6x8/bayes_mannwhitney4d26hpof/bayes_mannwhitney-20250423064402_1.csv', 'method=sample', 'algorithm=hmc', 'adapt', 'engaged=1']
 retcodes=[0, 0, 0, 0]
 per-chain output files (showing chain 1 only):
 csv_file:
	/tmp/tmpadqwb6x8/bayes_mannwhitney4d26hpof/bayes_mannwhitney-20250423064402_1.csv
 console_msgs (if any):
	/tmp/tmpadqwb6x8/bayes_mannwhitney4d26hpof/bayes_mannwhitney-20250423064402_0-stdout.txt
DEBUG:cmdstanpy:Chain 1 console:
method = sample (Default)
  sample
    num_samples = 1000 (Default)
    num_warmup = 1000 (Default)
    save_warmup = false (Default)
    thin = 1 (Default)
    adapt
      engaged = true (Default)
      gamm

In [23]:
pd.DataFrame(d)

,Pilot,BF01,p_greater
0,1,1.693040,0.03150
1,2,4.904226,0.81275
2,3,1.384584,0.02800
3,4,6.947966,0.39725
4,5,0.798964,0.02125


## Accumulated data

In [26]:
dic = {'BF01': [], 'p_greater': []}

group_A = df_habits[(df_habits['Group'] == 'ET') & (df_habits['Pilot'] != 5)]['dev_percent']
group_B = df_habits[(df_habits['Group'] == 'ST') & (df_habits['Pilot'] != 5)]['dev_percent']

  # Prepare the data dictionary for Stan
data = {
    'N_A': len(group_A),
    'N_B': len(group_B),
    'y_A': group_A,
    'y_B': group_B
}

fit = model.sample(data=data, chains=4, parallel_chains=4)

# Extract the posterior samples for the difference
diff_samples = fit.stan_variable('diff')

# Posterior KDE for the difference
kde = gaussian_kde(diff_samples)
x_range = np.linspace(-3, 3, 1000)
posterior_density = kde.evaluate(x_range)

# Prior for the difference (Normal(0, sqrt(2)))
prior_density = norm.pdf(x_range, loc=0, scale=np.sqrt(2))

# Bayes Factor using Savage-Dickey
p_post_0 = kde.evaluate(0)[0]
p_prior_0 = norm.pdf(0, loc=0, scale=np.sqrt(2))
BF_01 =  p_post_0 / p_prior_0

p_greater = np.mean(diff_samples > 0)

# Print Bayes Factor
dic['BF01'].append(BF_01)
dic['p_greater'].append(p_greater)

DEBUG:cmdstanpy:cmd: /content/drive/My Drive/Stans/bayes_mannwhitney info
cwd: None
DEBUG:cmdstanpy:input tempfile: /tmp/tmpadqwb6x8/0a8zsplo.json
06:44:52 - cmdstanpy - INFO - CmdStan start processing
INFO:cmdstanpy:CmdStan start processing


chain 1 |          | 00:00 Status

chain 2 |          | 00:00 Status

chain 3 |          | 00:00 Status

chain 4 |          | 00:00 Status

DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: 1
DEBUG:cmdstanpy:CmdStan args: ['/content/drive/My Drive/Stans/bayes_mannwhitney', 'id=1', 'random', 'seed=95630', 'data', 'file=/tmp/tmpadqwb6x8/0a8zsplo.json', 'output', 'file=/tmp/tmpadqwb6x8/bayes_mannwhitney_ko2ok2_/bayes_mannwhitney-20250423064452_1.csv', 'method=sample', 'algorithm=hmc', 'adapt', 'engaged=1']
DEBUG:cmdstanpy:idx 1
DEBUG:cmdstanpy:running CmdStan, num_threads: 1
DEBUG:cmdstanpy:CmdStan args: ['/content/drive/My Drive/Stans/bayes_mannwhitney', 'id=2', 'random', 'seed=95630', 'data', 'file=/tmp/tmpadqwb6x8/0a8zsplo.json', 'output', 'file=/tmp/tmpadqwb6x8/bayes_mannwhitney_ko2ok2_/bayes_mannwhitney-20250423064452_2.csv', 'method=sample', 'algorithm=hmc', 'adapt', 'engaged=1']
DEBUG:cmdstanpy:idx 2
DEBUG:cmdstanpy:running CmdStan, num_threads: 1
DEBUG:cmdstanpy:CmdStan args: ['/content/drive/My Drive/Stans/bayes_mannwhitney', 'id=3', 'random', 'seed=95630', 'data', 'file=/tmp/tmpadqwb6x8/0a8zsplo.js

06:44:52 - cmdstanpy - INFO - CmdStan done processing.
INFO:cmdstanpy:CmdStan done processing.
DEBUG:cmdstanpy:runset
RunSet: chains=4, chain_ids=[1, 2, 3, 4], num_processes=4
 cmd (chain 1):
	['/content/drive/My Drive/Stans/bayes_mannwhitney', 'id=1', 'random', 'seed=95630', 'data', 'file=/tmp/tmpadqwb6x8/0a8zsplo.json', 'output', 'file=/tmp/tmpadqwb6x8/bayes_mannwhitney_ko2ok2_/bayes_mannwhitney-20250423064452_1.csv', 'method=sample', 'algorithm=hmc', 'adapt', 'engaged=1']
 retcodes=[0, 0, 0, 0]
 per-chain output files (showing chain 1 only):
 csv_file:
	/tmp/tmpadqwb6x8/bayes_mannwhitney_ko2ok2_/bayes_mannwhitney-20250423064452_1.csv
 console_msgs (if any):
	/tmp/tmpadqwb6x8/bayes_mannwhitney_ko2ok2_/bayes_mannwhitney-20250423064452_0-stdout.txt
DEBUG:cmdstanpy:Chain 1 console:
method = sample (Default)
  sample
    num_samples = 1000 (Default)
    num_warmup = 1000 (Default)
    save_warmup = false (Default)
    thin = 1 (Default)
    adapt
      engaged = true (Default)
      gamm

In [27]:
pd.DataFrame(dic)

,BF01,p_greater
0,4.216294,0.0425
